# Backtest Visualizer

Loads a cached strategy by `CACHE_KEY`, re-runs its generated backtest, and plots:

1. **Equity curve** with drawdown shading
2. **Underwater (drawdown) plot**
3. **PnL distribution** (winners vs losers)
4. **Trades over time**
5. **Parameter optimization heatmap**

Set `CACHE_KEY` below. Find keys in `sources.yaml` or the `cache/` folder.
The multi-factor strategy fetches its own data via yfinance (no Alpaca/env needed);
intraday strategies load a cached parquet if available.

In [ ]:
import sys, json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Project root = parent of this notebooks/ folder
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CACHE_DIR = ROOT / 'cache'
OUTPUT_DIR = ROOT / 'output'
STRATEGIES_DIR = ROOT / 'strategies'

# ---- pick which strategy to visualize ----
CACHE_KEY = '0d87f719dc0c'   # Multi-Factor AI Analyst (self-contained, yfinance)

plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
import importlib.util

# Load the cached strategy config + generated backtest module
strategy_path = CACHE_DIR / f'{CACHE_KEY}_strategy.json'
code_path = CACHE_DIR / f'{CACHE_KEY}_backtest.py'
assert strategy_path.exists(), f'No strategy.json for {CACHE_KEY}'
assert code_path.exists(), f'No backtest.py for {CACHE_KEY}'

strategy = json.loads(strategy_path.read_text(encoding='utf-8'))

spec = importlib.util.spec_from_file_location(f'bt_{CACHE_KEY}', str(code_path))
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

DEFAULT_PARAMS = getattr(module, 'DEFAULT_PARAMS', {})
PARAM_GRID = getattr(module, 'PARAM_GRID', {})

print('Strategy :', strategy.get('name'))
print('Source   :', strategy.get('source_video'))
print('Params   :', DEFAULT_PARAMS)

In [ ]:
def load_market_data(strategy):
    """Return a cached price DataFrame for intraday strategies, or None.
    The multi-factor strategy ignores this and fetches its own data."""
    ticker_map = {'ES': 'SPY', 'NQ': 'QQQ', 'S&P 500': 'SPY', 'S&P500': 'SPY',
                  'NASDAQ': 'QQQ', 'RUSSELL': 'IWM', 'DOW': 'DIA'}
    instrument = (strategy.get('instrument', 'SPY') or 'SPY').upper()
    symbol = ticker_map.get(instrument, instrument.split()[0] if instrument else 'SPY')

    candidates = list(CACHE_DIR.glob(f'{symbol}_5min_*.parquet'))
    candidates += list(STRATEGIES_DIR.glob('*_5min_*.parquet'))
    candidates += list(STRATEGIES_DIR.glob('spy_5min_data.parquet'))
    if candidates:
        path = max(candidates, key=lambda p: p.stat().st_mtime)
        df = pd.read_parquet(path)
        print(f'Loaded market data: {path.name} ({len(df)} bars)')
        return df
    print('No cached parquet found; backtest must be self-contained (e.g. yfinance).')
    return None

data = load_market_data(strategy)

In [ ]:
# Run the backtest with default params -> trades DataFrame
trades = module.run_backtest(data, DEFAULT_PARAMS)
assert trades is not None and len(trades) > 0, 'No trades generated'

# Normalize an entry-date column and sort chronologically
date_col = 'date' if 'date' in trades.columns else trades.columns[0]
trades = trades.copy()
trades[date_col] = pd.to_datetime(trades[date_col])
trades = trades.sort_values(date_col).reset_index(drop=True)

pnls = trades['pnl'].values
winners = pnls[pnls > 0]
losers = pnls[pnls <= 0]
gp = winners.sum() if len(winners) else 0.0
gl = abs(losers.sum()) if len(losers) else 0.0

print(f"Trades        : {len(trades)}")
print(f"Win rate      : {len(winners) / len(pnls) * 100:.1f}%")
print(f"Total PnL     : {pnls.sum():.2f} points")
print(f"Profit factor : {gp / gl:.2f}" if gl else 'Profit factor : inf')
trades.head()

## 1. Equity curve & drawdown

In [ ]:
cum = np.cumsum(pnls)
peak = np.maximum.accumulate(cum)
dd = cum - peak
x = trades[date_col]

fig, ax = plt.subplots()
ax.plot(x, cum, color='#2563eb', lw=1.8, label='Cumulative PnL')
ax.fill_between(x, cum, peak, where=(cum < peak), color='red', alpha=0.15, label='Drawdown')
ax.axhline(0, color='gray', lw=0.8)
ax.set_title(f"Equity Curve — {strategy.get('name')}")
ax.set_ylabel('Cumulative PnL (points)')
ax.legend()
plt.tight_layout(); plt.show()

print(f'Max drawdown: {dd.min():.2f} points')

## 2. Underwater plot (drawdown over time)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3))
ax.fill_between(x, dd, 0, color='red', alpha=0.4)
ax.set_title('Underwater Plot')
ax.set_ylabel('Drawdown (points)')
plt.tight_layout(); plt.show()

## 3. PnL distribution

In [ ]:
fig, ax = plt.subplots()
bins = np.histogram_bin_edges(pnls, bins=30)
ax.hist(winners, bins=bins, color='#16a34a', alpha=0.8, label=f'Winners ({len(winners)})')
ax.hist(losers, bins=bins, color='#dc2626', alpha=0.8, label=f'Losers ({len(losers)})')
ax.axvline(0, color='gray', lw=0.8)
ax.axvline(pnls.mean(), color='black', ls='--', lw=1, label=f'Mean {pnls.mean():.2f}')
ax.set_title('Per-Trade PnL Distribution')
ax.set_xlabel('PnL (points)'); ax.set_ylabel('Trades')
ax.legend()
plt.tight_layout(); plt.show()

## 4. Trades over time

Each trade as a marker at its entry date, colored by outcome and sized by magnitude.
If trades carry a `ticker`, the y-axis splits by symbol; otherwise it shows PnL.

In [ ]:
fig, ax = plt.subplots()
colors = np.where(pnls > 0, '#16a34a', '#dc2626')
sizes = 20 + np.abs(pnls) / (np.abs(pnls).max() or 1) * 180

if 'ticker' in trades.columns and trades['ticker'].nunique() > 1:
    y = trades['ticker']
    ax.scatter(x, y, c=colors, s=sizes, alpha=0.7, edgecolors='none')
    ax.set_ylabel('Ticker')
else:
    ax.scatter(x, pnls, c=colors, s=sizes, alpha=0.7, edgecolors='none')
    ax.axhline(0, color='gray', lw=0.8)
    ax.set_ylabel('PnL (points)')
ax.set_title('Trades Over Time  (green = win, red = loss, size = magnitude)')
plt.tight_layout(); plt.show()

## 5. Parameter optimization heatmap

Loads `output/{CACHE_KEY}_report.json` if present; otherwise re-runs the
optimization grid via the pipeline's backtest stage. Profit factor across the
two most-varied parameters — look for a robust *plateau*, not a lone spike.

In [ ]:
def get_optimization_df():
    report_path = OUTPUT_DIR / f'{CACHE_KEY}_report.json'
    if report_path.exists():
        rec = json.loads(report_path.read_text(encoding='utf-8')).get('top_optimized', [])
        if rec:
            print('Using cached optimization from report.json')
            return pd.DataFrame(rec)
    print('No cached report; running optimization grid (this can take a while)...')
    from stages.backtest import run_optimization
    return run_optimization(module, data)

opt = get_optimization_df()
param_cols = [c for c in PARAM_GRID.keys() if c in opt.columns]
print('Varied params available:', param_cols)
opt.head()

In [ ]:
metric = 'profit_factor'
if len(param_cols) >= 2 and metric in opt.columns:
    # Pick the two params with the most distinct tested values
    ranked = sorted(param_cols, key=lambda c: opt[c].nunique(), reverse=True)
    pa, pb = ranked[0], ranked[1]
    pivot = opt.pivot_table(index=pa, columns=pb, values=metric, aggfunc='mean')

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', origin='lower')
    ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
    ax.set_xlabel(pb); ax.set_ylabel(pa)
    ax.set_title(f'{metric} across {pa} x {pb}')
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8)
    fig.colorbar(im, ax=ax, label=metric)
    plt.tight_layout(); plt.show()
else:
    print('Need >=2 varied params and a', metric, 'column to draw a heatmap.')
    if metric in opt.columns and param_cols:
        print(opt.sort_values(metric, ascending=False).head(10))